In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer 
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.calibration import calibration_curve

import xgboost as xgb
import optuna

In [6]:
#Loss Functions

def multiclass_brier(y_true, probs):
    y_onehot = np.eye(3)[y_true]
    return np.mean(np.sum((probs-y_onehot)**2, axis=1))

def rps_score(y_true, probs):

    actual = np.eye(3)[y_true]

    pred_cum = np.cumsum(probs[:, :-1], axis=1)
    actual_cum = np.cumsum(actual[:, :-1], axis=1)

    return np.mean(
        np.sum((pred_cum - actual_cum)**2, axis=1)
    )

def calibration_score_plot(y_test, probs, n_bins=10):

    calibration_results = {}

    plt.figure(figsize=(8, 6))

    for class_id, name in enumerate(
        ["Home win", "Draw", "Away win"]
    ):

        # Convert multiclass labels to binary for this class
        y_binary = (y_test == class_id).astype(int)

        prob_true, prob_pred = calibration_curve(
            y_binary,
            probs[:, class_id],
            n_bins=n_bins,
            strategy="uniform"
        )

        calibration_results[name] = {
            "predicted_probability": prob_pred,
            "observed_frequency": prob_true
        }

        plt.plot(
            prob_pred,
            prob_true,
            marker="o",
            label=name
        )

    # Perfect calibration line
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    plt.title("Calibration Curve: Football Win/Draw/Loss Model")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed frequency")
    plt.legend()
    plt.grid(True)
    plt.show()

    return calibration_results

In [7]:
features = pd.read_csv("stored_features/match_features_c_3_mu1_0.50_mu2_0.20.csv")

In [8]:
features = features.loc[(features["date"] < "2026-06-01") & (features["date"] > "1920-01-01")].sort_values(by="date", ascending=True)
#features = features.to_numpy()

In [10]:
print(features["winner_code"].value_counts())

winner_code
2    17556
1    16816
0    10389
Name: count, dtype: int64


In [ ]:
features.columns
features = features.drop(columns=["date", "home_team", "away_team", "home_score", "away_score", "winner", "home_avg_goal_difference_last_10", "away_avg_goal_difference_last_10","home_failed_score_rate_last_5", "away_failed_score_rate_last_5"])


In [ ]:
scaler = StandardScaler()
X_train, X_test, Y_train, Y_test = None, None, None, None

#TSS = TimeSeriesSplit(n_splits=5)
#split = TSS.split(features)
# for i, (train_index, test_index) in enumerate(split):
#     # print(f"Split {i+1}:")
#     # print(f"Train indices: {train_index}")
#     # print(f"Test indices: {test_index}")
#     X_train = features.iloc[train_index]
#     X_test = features.iloc[test_index]

# Y_train = X_train["winner_code"]
# Y_test = X_test["winner_code"]
# X_train = X_train.drop(columns=["winner_code"])
# X_test = X_test.drop(columns=["winner_code"])


y = features['winner_code']
X = features.drop(columns=["winner_code"])
cut_point = int(len(features) * 0.8)
X_train, X_test = X.iloc[:cut_point], X.iloc[cut_point:]
Y_train, Y_test = y.iloc[:cut_point], y.iloc[cut_point:]

imputer = SimpleImputer(strategy='most_frequent')
X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# X_train = X_train.to_numpy()
# X_test = X_test.to_numpy()
# Y_train = Y_train.to_numpy()
# Y_test = Y_test.to_numpy()

In [ ]:

def objective(trial):

    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "random_state": 42,

        # Boosting parameters
        "n_estimators": trial.suggest_int(
            "n_estimators",
            low=200,
            high=800,
            step=50
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            low=0.03,
            high=0.17,
            log=True
        ),

        # Tree structure parameters
        "max_depth": trial.suggest_int(
            "max_depth",
            low=2,
            high=6,
            step=1
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            low=1,
            high=15,
            step=1
        ),

        "gamma": trial.suggest_float(
            "gamma",
            low=0.0,
            high=5.0,
            step=0.1
        ),

        # Sampling parameters
        "subsample": trial.suggest_float(
            "subsample",
            low=0.6,
            high=1.0,
            step=0.05
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            low=0.6,
            high=1.0,
            step=0.05
        ),

        "colsample_bylevel": trial.suggest_float(
            "colsample_bylevel",
            low=0.6,
            high=1.0,
            step=0.05
        ),

        # Regularization parameters
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            low=1.0,
            high=20.0,
            log=True
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            low=0.0,
            high=5.0,
            step=0.1
        ),

        # Training parameters
        "tree_method": "hist",
        "n_jobs": -1
    }

    model = xgb.XGBClassifier(**params)

    model.fit(
        X_train,
        Y_train,
        eval_set=[
            (X_train, Y_train),
            (X_test, Y_test)
        ],
        verbose=False
    )


    probs = model.predict_proba(X_test)

    results ={
        "log_loss": log_loss(Y_test, probs),
        "brier_score": multiclass_brier(Y_test, probs),
        "rps_score": rps_score(Y_test, probs)
    }
    print("Trial: ", trial.number, " - RPS score: ", results["rps_score"], " - Brier score: ", results["brier_score"], " - Log loss: ", results["log_loss"], "\n")
    return results["rps_score"]

In [ ]:
study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=50
)

In [ ]:
print(study.best_params)

In [ ]:
best_params = study.best_params

final_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    **best_params
)

final_model.fit(
    X_train,
    Y_train,
    eval_set=[
            (X_train, Y_train),
            (X_test, Y_test)
        ],
        verbose=True)

In [ ]:
probs = final_model.predict_proba(X_test)

results ={
    "log_loss": log_loss(Y_test, probs),
    "brier_score": multiclass_brier(Y_test, probs),
    "rps_score": rps_score(Y_test, probs)
}

print(results)

output = calibration_score_plot(Y_test, probs)

In [ ]:
plot, ax = plt.subplots(figsize=(12, 12))
xgb.plot_importance(final_model, ax=ax)
plt.show()